In [ ]:
import os 
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
from typing import List
import time
import os
import numpy as np
import time
import os
import numpy as np
import torch
import pickle
import argparse
from uuid import uuid4

from torch.utils.data import DataLoader

import sys
sys.path.append('..')
sys.path.append('../..')
sys.path.append('../../..')

In [ ]:
import os
import pickle
import numpy as np
from torch.utils.data import DataLoader
from functools import partial
from argparse import ArgumentParser
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning import seed_everything
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from itertools import product
from glob import glob 
import re 

In [ ]:
from Model.backbone import generate_backbone
from Model.head import generate_head
from Model.model import MDNet

import yaml
from easydict import EasyDict

In [ ]:
import os
import torch
from   torch import nn
import torch.nn.functional as F
import pytorch_lightning as pl
from   pytorch_lightning.loggers.tensorboard import TensorBoardLogger
from   learnts.db import DB, my_collate_fn, num_images
from   learnts.model import MyModel
from torch_scatter import scatter_mean, scatter_add
from torch.optim import LBFGS

In [ ]:
import numpy as np
from ase import Atoms
from ase.calculators.calculator import (Calculator, CalculatorError, 
                    CalculatorSetupError, all_changes, all_properties, kpts2mp, FileIOCalculator)
from pyscf import gto, dft
from pyscf.geomopt.geometric_solver import optimize
from pyscf.hessian import thermo
import pyscf

AU2KCALMOL = 627.509608
AU2EV = 27.2114
BOHR = 0.52917721092

def ase_atoms_to_pyscf(ase_atoms):
    '''Convert ASE atoms to PySCF atom.

    Note: ASE atoms always use A.
    '''
    return [[atom.symbol, atom.position] for atom in ase_atoms]

atoms_from_ase = ase_atoms_to_pyscf

def calculate_efh(
    atom,
    f=True,
    hess=False,
    return_metrics=False,
    xc="wb97x",
    basis="631g*",
    device='gpu',
    # d3=False,
):
    geomfile = ase_atoms_to_pyscf(atom)
    spin = 0
    mol = pyscf.M(
        atom=geomfile,
        unit="Ang",
        basis=basis,
    )
    mol.build()

    if device == 'gpu':
        mf = dft.RKS(mol).to_gpu() if not spin else dft.UKS(mol).to_gpu()
    else:
        mf = dft.RKS(mol) if not spin else dft.UKS(mol)
    mf.xc = xc
    # if d3:
    #     mf = dftd3.dftd3(dft.RKS(mol, xc=xc))
    mf.conv_tol = 1e-6
    # mf.damp = 0.2
    mf.max_cycle = 200
    mf.max_memory = 32000
    mf.run()

    force = None
    force_rms = np.nan
    if mf.converged and f:
        force = mf.nuc_grad_method().kernel() * -1.0 / BOHR
        force_rms = np.sqrt(np.mean(force**2)) * AU2EV
        print("force rms (ev/A): ", force_rms)

    hessian = None
    if hess:
        hessian = mf.Hessian().kernel()
        freq_info = thermo.harmonic_analysis(mf.mol, hessian)
        print("freq: ", freq_info["freq_wavenumber"])

    if return_metrics:
        return (
            mf,
            force,
            hessian,
            force_rms,
            count_negative_eig(freq_info["freq_wavenumber"]),
        )
    return mf, force, hessian

# K-point sampling
def make_kpts(cell, nks):
    raise DeprecationWarning('Use cell.make_kpts(nks) instead.')

def count_negative_eig(x: list):
    count = 0
    for _x in x:
        if _x.imag > 0:
            count += 1
    return count

In [ ]:
def calculate_loss_coord(adj_matrix, coord):
    diff = coord[:, torch.newaxis, :] - coord[torch.newaxis, :, :]
    dists = torch.norm(diff, p=2, dim=-1)
    loss = torch.sum(torch.exp(-0.2 * (adj_matrix ** 2)) * torch.square(dists - adj_matrix))
    return loss

def Kabsch_alignment(pos_to_fit, pos, batch):
    v = pos.shape[-1]
    center = scatter_mean(pos, batch, dim = -2) # B * 3
    pos_to_fit_center = scatter_mean(pos_to_fit, batch, dim = -2) # B * 3
    pos_c = pos - center[batch]
    pos_to_fit_c = pos_to_fit - pos_to_fit_center[batch]
    pos_c = pos_c.repeat([1,v])
    pos_to_fit_c = pos_to_fit.repeat_interleave(v,dim=-1)
    H = scatter_add(pos_c * pos_to_fit_c, batch, dim = -2).reshape(-1,v,v) # B * 3 * 3
    U, S, V = torch.svd(H)
    I = torch.eye(3).unsqueeze(0).repeat(H.shape[0], 1, 1)
    temp = torch.linalg.det(V @ U.transpose(2,1))
    d = torch.ones_like(temp)
    mask = temp < 0
    d[mask] *= -1
    I[:, -1, -1] = d
    # Rotation matrix
    R = V @ U.transpose(2,1)
    t = center - (pos_to_fit_center.unsqueeze(1) @ R.transpose(2,1)).squeeze(1)
    R = R[batch]
    t = t[batch]
    p_aligned = (pos_to_fit.unsqueeze(1) @ R.transpose(2,1)).squeeze(1) + t
    return p_aligned

def rmsd_loss(pred, target, batch):
    temp = torch.nn.functional.mse_loss(pred, target, reduction='none')
    temp = torch.sum(temp, dim=-1)
    temp = scatter_mean(temp, batch)
    temp = torch.sqrt(temp)
    return temp

def d_mae_loss(pred, target, batch):
    device = batch.device
    src = torch.arange(0, batch.shape[0])
    dst = torch.arange(0, batch.shape[0])
    src = torch.repeat_interleave(src, batch.shape[0])
    dst = dst.repeat(batch.shape[0])
    mask = (batch[src] == batch[dst])
    src = src.to(device)
    dst = dst.to(device)
    src = src[mask]
    dst = dst[mask]
    mask = (src != dst)
    src = src[mask]
    dst = dst[mask]
    diff_pred = torch.norm(pred[src] - pred[dst], p=2, dim=-1)
    diff_true = torch.norm(target[src] - target[dst], p=2, dim=-1)
    loss = scatter_mean(torch.abs(diff_pred - diff_true), batch[src])
    return loss

In [ ]:
parser = argparse.ArgumentParser(description='Training Transition1x dynamics')
parser.add_argument('--config_file', required=True)
parser.add_argument('--log_prefix', default='logs')
parser.add_argument('--notes', default=' ')
parser.add_argument('--device', default='cuda')
parser.add_argument('--resume_status', default=' ')
parser.add_argument('--potential', default=' ')
parser.add_argument('--checkpoint', default=' ')
parser.add_argument('--hdf5_file', default=' ')
parser.add_argument('--batch_size', type=int, default=1)
args = parser.parse_args(['--config_file', "../Configs/Potential.yml",
                          '--device', 'cpu',
                          '--hdf5_file', '', # path to transition1x.h5 file
                          '--checkpoint', '', # path to checkpoint file
                          '--potential', '']) # path to trained MLIP checkpoint

In [ ]:
dtype = torch.float32

config_path=args.config_file
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)
config = EasyDict(config)
config.notes = args.notes

device = args.device
backbone = generate_backbone(config.model.backbone)
head = generate_head(config.model.head)

config.data.batch_size = 1

In [ ]:
REFERENCE_ENERGIES = {
    1: -13.62222753701504,
    6: -1029.4130839658328,
    7: -1484.8710358098756,
    8: -2041.8396277138045,
    9: -2712.8213146878606,
}

In [ ]:
potential_model = MDNet(backbone, head, REFERENCE_ENERGIES)
best_state = torch.load(args.potential, map_location=device)
potential_model.load_state_dict(best_state['model'])
potential_model.eval()

In [ ]:
potential_model = potential_model.to(device)

In [ ]:
grid = np.mgrid[0.0:30.0:301j]
db = DB(args.hdf5_file, 'test')
loader = DataLoader(db, batch_size = config.data.batch_size, shuffle=False, collate_fn=partial(my_collate_fn, num_images=num_images, grid = grid, alpha=10.0, duplicate=True, test=True))

words = re.split('-',re.split('/', args.checkpoint)[-4])
learning_rate = float(words[-2]+'-'+words[-1])
factor1 = float(words[4])
factor2 = float(words[5])
num_layers = int(words[1])
num_gru_layers = int(words[2])
gru_dropout = float(words[3])
num_pair_embedding = int(words[0])

In [ ]:
model = MyModel.load_from_checkpoint(checkpoint_path=args.checkpoint).to(device)
model.TTA = True # or False
model = model.to(device)

In [ ]:
reactant_product_pos = []
transition_state_pos_true = []
transition_state_pos_pred = []
true_trans_energy_list = []
atomic_number_list = []

In [ ]:
total_loss_rmsd = []
total_loss_dmae = []
total_loss_energy_r = []
total_loss_energy_dft = []
cnt = 0

for data in loader:
    dist_mat, pred_e_r, pred_e_p = model.test_step(data, None)
    true_pos_tran = data[-2]
    true_pos_r = data[-4]
    true_pos_p = data[-3]
    true_trans_energy = data[-1]
    atomic_numbers = data[0][0]
    true_energy_r = data[4]
    true_energy_p = data[5]

    reactant_product_pos.append((true_pos_r, true_pos_p))
    transition_state_pos_true.append(true_pos_tran)
    atomic_number_list.append(atomic_numbers)
    true_trans_energy_list.append(true_trans_energy)

    pred_pos = torch.tensor((true_pos_p + true_pos_r) / 2, dtype=dist_mat[0].dtype, requires_grad=True)
    optimizer = LBFGS([pred_pos], max_iter=100, lr=0.1)
    def closure():
        optimizer.zero_grad()
        loss = calculate_loss_coord(dist_mat[0], pred_pos)
        loss.backward()
        return loss
    optimizer.step(closure)
    loss_pos = calculate_loss_coord(dist_mat[0], pred_pos)

    pred_pos = Kabsch_alignment(pred_pos, true_pos_tran, torch.zeros(pred_pos.shape[0], dtype=torch.int64))

    transition_state_pos_pred.append(pred_pos)

    loss_rmsd = rmsd_loss(pred_pos, true_pos_tran, torch.zeros(pred_pos.shape[0], dtype=torch.int64))
    loss_dmae = d_mae_loss(pred_pos, true_pos_tran, torch.zeros(pred_pos.shape[0], dtype=torch.int64))
    energy_r, _ = potential_model.get_energy_and_force(atomic_numbers, true_pos_r, None, None, torch.zeros_like(atomic_numbers, device=device))
    energy_t, _ = potential_model.get_energy_and_force(atomic_numbers, pred_pos, None, None, torch.zeros_like(atomic_numbers, device=device))
    # loss_energy_r = torch.abs(energy_t - energy_r - true_energy_r[0])
    atoms = Atoms(numbers=atomic_numbers.cpu().numpy(), positions=pred_pos.detach().cpu().numpy())
    # energy_dft = calculate_efh(atoms, f=True)[0].e_tot * AU2EV
    energy_dft = 0.0
    loss_energy_dft = torch.abs(energy_dft - true_trans_energy)
    loss_energy_r = torch.abs(energy_t - true_trans_energy)
    total_loss_rmsd.append(loss_rmsd.detach().numpy())
    total_loss_dmae.append(loss_dmae.detach().numpy())
    total_loss_energy_r.append(loss_energy_r.detach().cpu().numpy())
    total_loss_energy_dft.append(loss_energy_dft.detach().cpu().numpy())
print('avg: ', np.mean(total_loss_rmsd), np.mean(total_loss_dmae), np.mean(total_loss_energy_r), np.mean(total_loss_energy_dft))
print('med: ', np.median(total_loss_rmsd), np.median(total_loss_dmae), np.median(total_loss_energy_r), np.median(total_loss_energy_dft))

In [ ]:
import pickle
with open('res_learnts.pickle', 'wb') as f:
    pickle.dump({
        'rmsd': total_loss_rmsd,
        'dmae': total_loss_dmae,
        'reactant_product_pos': reactant_product_pos,
        'atom_types': atomic_number_list,
        'true_trans_energy': true_trans_energy_list,
        'pred_transition_state_pos': transition_state_pos_pred,
        'true_transition_state_pos': transition_state_pos_true
    }, f)